# Building an Autograd Engine from Scratch

How does PyTorch compute gradients? In this notebook we'll build a tiny autograd engine — the same idea that powers PyTorch, TensorFlow, and JAX.

**Three ways to compute derivatives:**
1. **By hand** (symbolic) — exact but tedious, doesn't scale
2. **Finite differences** — approximate, easy to code, but slow and imprecise
3. **Automatic differentiation (autograd)** — exact AND automatic

We'll see why finite differences fail, then build autograd from scratch.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 12
print('Ready!')

---
## Part 1: What AutoDiff is NOT — Finite Differences

The simplest idea: wiggle each input by a tiny amount $h$ and see how the output changes.

**One-sided:**
$$\frac{\partial f}{\partial x_i} \approx \frac{f(x_1, \ldots, x_i + h, \ldots) - f(x_1, \ldots, x_i, \ldots)}{h}$$

**Two-sided (central difference):**
$$\frac{\partial f}{\partial x_i} \approx \frac{f(x_1, \ldots, x_i + h, \ldots) - f(x_1, \ldots, x_i - h, \ldots)}{2h}$$

In [ ]:
# Simple example: f(x) = x^2 at x = 3
# Exact derivative: f'(x) = 2x = 6

def f(x):
    return x**2

x = 3.0
h = 0.001

# One-sided
one_sided = (f(x + h) - f(x)) / h

# Two-sided (central difference)
two_sided = (f(x + h) - f(x - h)) / (2 * h)

print(f'Exact derivative:    f\'(3) = 2*3 = 6.0')
print(f'One-sided (h=0.001): {one_sided:.6f}  (error: {abs(one_sided - 6.0):.6f})')
print(f'Two-sided (h=0.001): {two_sided:.6f}  (error: {abs(two_sided - 6.0):.6f})')

### Challenges with Finite Differences

1. **Expensive:** Need a forward pass for EACH variable
2. **Numerically unstable:** Choice of $h$ is tricky

In [ ]:
# The Goldilocks problem: h too big = bad approximation, h too small = floating point errors
h_values = np.logspace(-15, 0, 100)
errors = [abs((f(x + h) - f(x - h)) / (2 * h) - 6.0) for h in h_values]

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(h_values, errors, 'b-', linewidth=2)
ax.axvline(x=1e-5, color='green', linestyle=':', alpha=0.7, label='h = 1e-5 (sweet spot)')
ax.set_xlabel('Step size h')
ax.set_ylabel('Absolute error')
ax.set_title('Finite Differences: The Goldilocks Problem', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('Too large h → poor approximation')
print('Too small h → floating-point errors dominate')
print('You can NEVER get an exact answer with finite differences!')

In [ ]:
# Cost: need 2 forward passes PER parameter
n_params = [3, 100, 1_000_000, 175_000_000_000]
names = ['Toy example', 'Small model', 'ResNet-50', 'GPT-3']

print('Cost of finite differences:')
print(f'{"Model":<20} {"Parameters":>15} {"Forward passes":>20}')
print('-' * 57)
for name, n in zip(names, n_params):
    print(f'{name:<20} {n:>15,} {2*n:>20,}')

print()
print('GPT-3 would need 350 BILLION forward passes for one gradient step!')
print('Autograd does it in ONE backward pass.')

---
## Part 2: Computational Graphs

The key idea behind autograd:
- **Nodes:** Operations (+, ×, exp, ...)
- **Edges:** Variables / Tensors (and data dependencies)

Example: $(x \cdot y) + z$

```
x ──→ [×] ──→ [+] ──→ output
y ──↗         ↑
z ─────────────┘
```

Every computation can be broken into elementary operations, each of which knows its own local gradient.

### Backprop Through a Computational Graph

For a node $z = f(x, y)$ in a larger computation ending at loss $J$:

$$\frac{dJ}{dx} = \underbrace{\frac{dJ}{dz}}_{\text{upstream gradient}} \cdot \underbrace{\frac{dz}{dx}}_{\text{local gradient}}$$

$$\boxed{\text{downstream gradient} = \text{upstream gradient} \times \text{local gradient}}$$

That's it! Each node multiplies the incoming gradient by its local gradient and passes it backward.

---
## Part 3: Building the Value Class

Let's build a `Value` class that:
1. Wraps a scalar number
2. Tracks the computation graph (which operations created it)
3. Can compute gradients automatically

We'll start with just the forward pass.

### Step 1: Forward pass only (just `__add__`)

In [ ]:
class Value:
    """Stores a single scalar value and its gradient."""
    
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        return out
    
    def __repr__(self):
        return f"Value(data={self.data})"

In [ ]:
x = Value(1.0)
y = Value(2.0)
z = x + y

print(f'x = {x}')
print(f'y = {y}')
print(f'z = x + y = {z}')
print(f'z._prev = {z._prev}')  # tracks its parents!
print(f'z._op = "{z._op}"')    # tracks the operation!

### Step 2: Add multiplication, build our first expression

In [ ]:
class Value:
    """Stores a single scalar value and its gradient."""
    
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        return out
    
    def __repr__(self):
        return f"Value(data={self.data})"

In [ ]:
# Our primary example: x=2, y=-3, mul=x*y, b=10, L=mul+b
x = Value(2.0)
y = Value(-3.0)
mul = x * y        # mul = 2 * (-3) = -6
b = Value(10.0)
L = mul + b         # L = -6 + 10 = 4

print(f'x   = {x}')
print(f'y   = {y}')
print(f'mul = x * y = {mul}')
print(f'b   = {b}')
print(f'L   = mul + b = {L}')

The computation graph looks like:
```
x (2) ──→ [*] ──→ mul (-6) ──→ [+] ──→ L (4)
y (-3) ─↗                       ↑
                          b (10) ┘
```

Now let's add the ability to compute gradients!

---
## Part 4: Adding Backward (Gradient Computation)

Each operation needs to know its **local gradient**:

| Operation | Output | Local gradient |
|-----------|--------|----------------|
| `c = a + b` | `a + b` | $\frac{\partial c}{\partial a} = 1$, $\frac{\partial c}{\partial b} = 1$ |
| `c = a * b` | `a × b` | $\frac{\partial c}{\partial a} = b$, $\frac{\partial c}{\partial b} = a$ |

And the rule: **downstream = upstream × local**

### Manual backward pass on our example

Let's trace through by hand first.

In [ ]:
# Manual backward pass
# L = mul + b, where mul = x * y

# Step 1: dL/dL = 1 (gradient of output w.r.t. itself)
dL_dL = 1.0
print(f'dL/dL = {dL_dL}')

# Step 2: L = mul + b (addition)
# Local gradients: dL/d(mul) = 1, dL/db = 1
dL_dmul = dL_dL * 1.0  # upstream * local
dL_db = dL_dL * 1.0    # upstream * local
print(f'dL/d(mul) = {dL_dL} × 1 = {dL_dmul}')
print(f'dL/db     = {dL_dL} × 1 = {dL_db}')

# Step 3: mul = x * y (multiplication)
# Local gradients: d(mul)/dx = y = -3, d(mul)/dy = x = 2
x_val, y_val = 2.0, -3.0
dL_dx = dL_dmul * y_val  # upstream * local
dL_dy = dL_dmul * x_val  # upstream * local
print(f'dL/dx     = {dL_dmul} × y = {dL_dmul} × {y_val} = {dL_dx}')
print(f'dL/dy     = {dL_dmul} × x = {dL_dmul} × {x_val} = {dL_dy}')

print()
print('Summary:')
print(f'  dL/dx = {dL_dx}  (if we increase x by 1, L decreases by 3)')
print(f'  dL/dy = {dL_dy}   (if we increase y by 1, L increases by 2)')
print(f'  dL/db = {dL_db}   (if we increase b by 1, L increases by 1)')

### Now let's automate this in the Value class

In [ ]:
class Value:
    """Stores a single scalar value and its gradient."""
    
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0  # gradient starts at 0
        self._backward = lambda: None  # default: do nothing
        self._prev = set(_children)
        self._op = _op
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += out.grad   # local grad for + is 1
            other.grad += out.grad  # local grad for + is 1
        out._backward = _backward
        
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad  # local grad: other.data
            other.grad += self.data * out.grad   # local grad: self.data
        out._backward = _backward
        
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f'**{other}')
        
        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward
        
        return out
    
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

In [ ]:
# Test: manually call _backward step by step
x = Value(2.0)
y = Value(-3.0)
mul = x * y
b = Value(10.0)
L = mul + b

# Start backprop: dL/dL = 1
L.grad = 1.0

# Backward through +
L._backward()

# Backward through *
mul._backward()

print(f'x:   data={x.data}, grad={x.grad}')    # grad should be -3
print(f'y:   data={y.data}, grad={y.grad}')     # grad should be 2
print(f'mul: data={mul.data}, grad={mul.grad}') # grad should be 1
print(f'b:   data={b.data}, grad={b.grad}')     # grad should be 1
print(f'L:   data={L.data}, grad={L.grad}')     # grad should be 1
print()
print('Matches our manual computation!')

### Automating the backward pass with topological sort

Calling `_backward()` manually in the right order is tedious. Let's automate it:
1. Topologically sort the graph
2. Walk nodes in reverse order, calling `_backward()` on each

In [ ]:
class Value:
    """Stores a single scalar value and its gradient."""
    
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        # Topological sort
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # Backward pass
        self.grad = 1
        for v in reversed(topo):
            v._backward()
    
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

In [ ]:
# Now just one call: L.backward()
x = Value(2.0)
y = Value(-3.0)
mul = x * y
b = Value(10.0)
L = mul + b

L.backward()  # computes ALL gradients automatically!

print(f'x:   data={x.data:6.1f}, grad={x.grad:6.1f}')
print(f'y:   data={y.data:6.1f}, grad={y.grad:6.1f}')
print(f'mul: data={mul.data:6.1f}, grad={mul.grad:6.1f}')
print(f'b:   data={b.data:6.1f}, grad={b.grad:6.1f}')
print(f'L:   data={L.data:6.1f}, grad={L.grad:6.1f}')

---
## Part 5: Visualizing the Computation Graph

Let's draw the graph showing data and gradients at each node.

In [ ]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})
    
    for n in nodes:
        dot.node(name=str(id(n)), 
                 label="{ data %.4f | grad %.4f }" % (n.data, n.grad), 
                 shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [ ]:
# Visualize our example!
x = Value(2.0)
y = Value(-3.0)
mul = x * y
b = Value(10.0)
L = mul + b
L.backward()

draw_dot(L)

In [ ]:
# Another example: w*x + 3, squared
w = Value(5.0)
x = Value(1.0)
c = w * x + 3
d = c ** 2

d.backward()
draw_dot(d)

### Verify with finite differences

In [ ]:
# Let's verify our autograd gives the same answer as finite differences
h = 1e-5

# f(x, y) = x*y + 10  at x=2, y=-3
def f(x, y): return x * y + 10

dfdx_fd = (f(2 + h, -3) - f(2 - h, -3)) / (2 * h)
dfdy_fd = (f(2, -3 + h) - f(2, -3 - h)) / (2 * h)

# Our autograd values (from above)
x = Value(2.0)
y = Value(-3.0)
L = x * y + Value(10.0)
L.backward()

print(f'{"Method":<25} {"dL/dx":>10} {"dL/dy":>10}')
print('=' * 47)
print(f'{"Finite differences":<25} {dfdx_fd:>10.6f} {dfdy_fd:>10.6f}')
print(f'{"Our autograd":<25} {x.grad:>10.6f} {y.grad:>10.6f}')
print(f'{"Exact (by hand)":<25} {-3.0:>10.6f} {2.0:>10.6f}')
print()
print('Our autograd is exact! Finite differences is approximate.')

---
## Part 6: More Operations — Sigmoid, Negation, Subtraction, Division

We can build complex operations from primitives:
- **Negation:** `-a = a * (-1)` 
- **Subtraction:** `a - b = a + (-b)`
- **Division:** `a / b = a * (b ** -1)`
- **Sigmoid:** $\sigma(x) = \frac{1}{1 + e^{-x}}$, with derivative $\sigma(x)(1 - \sigma(x))$

In [ ]:
class Value:
    """Stores a single scalar value and its gradient."""
    
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward
        return out
    
    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.data))
        out = Value(s, (self,), 'sigmoid')
        def _backward():
            self.grad += (out.data * (1 - out.data)) * out.grad
        out._backward = _backward
        return out
    
    def log(self):
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()
    
    def __neg__(self):        return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

In [ ]:
# Quick test: sigmoid
a = Value(1.0)
b = 2 * a
c = b.sigmoid()

c.backward()
draw_dot(c)

---
## Part 7: Complex Example

Let's stress-test our autograd with a complex expression.

In [ ]:
x = Value(-4.0)
y = Value(2.0)

# A complex computation
q = x + y
r = x * y + y**3
q += q + 1
q += 1 + q + (-x)
r += r * 2 + (y + x).sigmoid()
r += 3 * r + (y - x).sigmoid()
s = q - r
t = s**2
u = t / 2.0
v = u + 10.0 / t

print(f'Final value: v.data = {v.data:.4f}')

v.backward()

print(f'x.grad = {x.grad:.4f}')
print(f'y.grad = {y.grad:.4f}')

draw_dot(v)

---
## Part 8: Analogy with PyTorch Tensors

Our `Value` class is a tiny version of `torch.Tensor`. Let's verify our results match PyTorch.

In [ ]:
# PyTorch equivalent of our primary example
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(-3.0, requires_grad=True)
b = torch.tensor(10.0, requires_grad=True)

mul = x * y
L = mul + b
L.backward()

print('PyTorch autograd:')
print(f'  x.grad = {x.grad.item()}  (our Value: -3.0)')
print(f'  y.grad = {y.grad.item()}   (our Value: 2.0)')
print(f'  b.grad = {b.grad.item()}   (our Value: 1.0)')
print()
print('Exact match! Our autograd engine works the same way as PyTorch.')

In [ ]:
# PyTorch: w*x + b example with gradient descent update
w = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(2.0, requires_grad=False)
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b       # y = 3*2 + 1 = 7
loss = y ** 2        # loss = 49

loss.backward()

print(f'y = w*x + b = {y.item()}')
print(f'loss = y² = {loss.item()}')
print(f'dloss/dw = {w.grad.item()}  (= 2*y*x = 2*7*2 = 28)')
print(f'dloss/db = {b.grad.item()}  (= 2*y*1 = 2*7 = 14)')
print()

# Gradient descent update
lr = 0.01
with torch.no_grad():
    w.data -= lr * w.grad
    b.data -= lr * b.grad

print(f'After one gradient descent step (lr={lr}):')
print(f'  w: 3.0 → {w.data.item():.4f}')
print(f'  b: 1.0 → {b.data.item():.4f}')

# Important: zero gradients!
w.grad.zero_()
b.grad.zero_()
print()
print('Always call zero_grad() before the next backward pass!')

In [ ]:
# Verify our complex example against PyTorch
x_v = Value(-4.0)
y_v = Value(2.0)
q = x_v + y_v; r = x_v * y_v + y_v**3
q += q + 1; q += 1 + q + (-x_v)
r += r * 2 + (y_v + x_v).sigmoid()
r += 3 * r + (y_v - x_v).sigmoid()
s = q - r; t = s**2; u = t / 2.0; v_val = u + 10.0 / t
v_val.backward()

x_t = torch.tensor(-4.0, requires_grad=True)
y_t = torch.tensor(2.0, requires_grad=True)
q = x_t + y_t; r = x_t * y_t + y_t**3
q = q + q + 1; q = q + 1 + q + (-x_t)
r = r + r * 2 + torch.sigmoid(y_t + x_t)
r = r + 3 * r + torch.sigmoid(y_t - x_t)
s = q - r; t = s**2; u = t / 2.0; v_torch = u + 10.0 / t
v_torch.backward()

print(f'{"":<20} {"Our Value":>15} {"PyTorch":>15} {"Match?":>8}')
print('=' * 60)
print(f'{"x.grad":<20} {x_v.grad:>15.4f} {x_t.grad.item():>15.4f} {"Yes" if abs(x_v.grad - x_t.grad.item()) < 1e-6 else "No":>8}')
print(f'{"y.grad":<20} {y_v.grad:>15.4f} {y_t.grad.item():>15.4f} {"Yes" if abs(y_v.grad - y_t.grad.item()) < 1e-6 else "No":>8}')

---
## Part 9: Logistic Regression Loss — A Real Computation Graph

Let's build the computation graph for logistic regression loss, step by step — exactly as we'd draw it on the board.

$$\hat{y} = \frac{1}{1 + e^{-(\theta_0 + \theta_1 x_1 + \theta_2 x_2)}}$$

For $y = 1$:
$$\text{Loss} = -\log(\hat{y}) = -1 \times \log\left(\frac{1}{1 + e^{-(\theta_0 + \theta_1 x_1 + \theta_2 x_2)}}\right)$$

Breaking into elementary ops:

$$\theta_1, x_1 \xrightarrow{\times} f_1 \quad\quad \theta_2, x_2 \xrightarrow{\times} f_2$$
$$f_1, f_2 \xrightarrow{+} f_3 \xrightarrow{+\theta_0} f_4 \xrightarrow{\times(-1)} f_5 \xrightarrow{\exp} f_6 \xrightarrow{+1} f_7 \xrightarrow{1/x} f_8 \xrightarrow{\log} f_9 \xrightarrow{\times(-1)} L$$

In [ ]:
# Logistic regression loss built from elementary operations
# Using the values from the slides: θ₁=1, x₁=1, θ₂=2, x₂=2, θ₀=1

theta1 = Value(1.0)
x1 = Value(1.0)
theta2 = Value(2.0)
x2 = Value(2.0)
theta0 = Value(1.0)

# Build the computation graph step by step
f1 = theta1 * x1           # θ₁ * x₁ = 1
f2 = theta2 * x2           # θ₂ * x₂ = 4
f3 = f1 + f2               # θ₁x₁ + θ₂x₂ = 5
f4 = f3 + theta0           # + θ₀ = 6
f5 = f4 * (-1)             # ×(-1) = -6
f6 = f5.exp()              # exp(-6)
f7 = f6 + 1                # 1 + exp(-6)
f8 = 1 / f7                # 1 / (1 + exp(-6)) = σ(6)
f9 = f8.log()              # log(σ(6))
L = f9 * (-1)              # -log(σ(6)) = Loss

print('Forward pass (step by step):')
print(f'  f1 = θ₁ × x₁         = {f1.data:.6f}')
print(f'  f2 = θ₂ × x₂         = {f2.data:.6f}')
print(f'  f3 = f1 + f2          = {f3.data:.6f}')
print(f'  f4 = f3 + θ₀          = {f4.data:.6f}')
print(f'  f5 = f4 × (-1)        = {f5.data:.6f}')
print(f'  f6 = exp(f5)          = {f6.data:.6f}')
print(f'  f7 = f6 + 1           = {f7.data:.6f}')
print(f'  f8 = 1/f7 = σ(6)      = {f8.data:.6f}')
print(f'  f9 = log(f8)          = {f9.data:.6f}')
print(f'  L  = f9 × (-1)        = {L.data:.6f}')
print()
print(f'Loss = {L.data:.6f}')

In [ ]:
# Backward pass — autograd does all the chain rule for us!
L.backward()

print('Gradients (computed automatically!):')
print(f'  dL/dθ₁ = {theta1.grad:.6f}')
print(f'  dL/dθ₂ = {theta2.grad:.6f}')
print(f'  dL/dθ₀ = {theta0.grad:.6f}')
print(f'  dL/dx₁ = {x1.grad:.6f}')
print(f'  dL/dx₂ = {x2.grad:.6f}')
print()

# Visualize the full computation graph
draw_dot(L)

In [ ]:
# Verify against PyTorch
t1 = torch.tensor(1.0, requires_grad=True)
t2 = torch.tensor(2.0, requires_grad=True)
t0 = torch.tensor(1.0, requires_grad=True)
tx1 = torch.tensor(1.0)
tx2 = torch.tensor(2.0)

z = t1 * tx1 + t2 * tx2 + t0
loss_pt = -torch.log(torch.sigmoid(z))
loss_pt.backward()

print('Verification against PyTorch:')
print(f'{"Parameter":<10} {"Our Value":>12} {"PyTorch":>12} {"Match?":>8}')
print('=' * 44)
for name, ours, theirs in [
    ('θ₁', theta1.grad, t1.grad.item()),
    ('θ₂', theta2.grad, t2.grad.item()),
    ('θ₀', theta0.grad, t0.grad.item()),
]:
    match = 'Yes' if abs(ours - theirs) < 1e-5 else 'No'
    print(f'{name:<10} {ours:>12.6f} {theirs:>12.6f} {match:>8}')

print()
print('Our hand-built autograd matches PyTorch exactly!')

---
## Summary

| Method | Exact? | Cost | Scales? |
|--------|--------|------|--------|
| By hand (symbolic) | Yes | Human time | No way |
| Finite differences | No (≈) | 2N forward passes | Very slow |
| **Autograd** | **Yes** | **1 backward pass** | **Yes!** |

**How autograd works:**
1. Forward pass records a computation graph
2. Each operation knows its own local gradient
3. Backward pass walks the graph in reverse, applying: **downstream = upstream × local**
4. Result: exact gradients for ALL parameters in one pass

**This is why deep learning is possible.** Without autograd, training a million-parameter network would require millions of forward passes per gradient step.

Our tiny `Value` class is exactly what PyTorch's `torch.Tensor` does — just for scalars instead of tensors!